# Stratified Corpus Quality Check — 5 Positions Across the Full Corpus

Answers: "does the 5,000-document sample represent the whole 1,800,000-document
corpus, or could quality/NL-content vary by position?"

Instead of one 5,000-doc block from the start, this pulls **5 smaller samples
(1,000 docs each) spread across the corpus** — start, 25%, 50%, 75%, near-end —
and runs the full quality audit on each slice independently. If all five give
similar numbers, the original 5,000-doc sample is confirmed representative.
If they diverge, that's a real finding worth reporting.

Runtime: ~10-15 min total (5 slices x 1,000 docs, CPU only, no GPU needed).


In [1]:
!pip install -q -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.9 MB/s eta 0:00:00


In [2]:
# ======================= CONFIG =======================
DATASET_ID   = "Ananda100/python-clean-codeparrot"
SPLIT        = "train"
SLICE_SIZE   = 1000        # docs per position
OUT_JSON     = "stratified_quality_report.json"
# ======================================================
print(f"Stratified audit of {DATASET_ID} [{SPLIT}]")

Stratified audit of Ananda100/python-clean-codeparrot [train]


## Definitions (same metrics as the main audit)

In [3]:
import ast, hashlib, re, statistics, warnings
from collections import Counter

warnings.filterwarnings("ignore", category=SyntaxWarning)


def nl_fraction(source):
    total = len(source)
    comment_chars = 0
    for line in source.split("\n"):
        idx = line.find("#")
        if idx != -1:
            comment_chars += len(line[idx:])
    docstring_chars = 0
    try:
        tree = ast.parse(source)
        for node in ast.walk(tree):
            if isinstance(node, (ast.Module, ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                doc = ast.get_docstring(node)
                if doc:
                    docstring_chars += len(doc)
    except SyntaxError:
        pass
    nl = comment_chars + docstring_chars
    return {"comment_chars": comment_chars, "docstring_chars": docstring_chars,
            "code_chars": max(total - nl, 0), "total_chars": total}


def is_valid_python(s):
    try:
        ast.parse(s)
        return True
    except SyntaxError:
        return False


def normalize_code(s):
    lines = [re.sub(r"#.*$", "", l).strip() for l in s.split("\n")]
    return "\n".join(l for l in lines if l)


def audit_slice(sources, label):
    n = len(sources)
    valid = sum(is_valid_python(s) for s in sources)
    tot = {"comment_chars": 0, "docstring_chars": 0, "total_chars": 0}
    for s in sources:
        r = nl_fraction(s)
        for k in ("comment_chars", "docstring_chars", "total_chars"):
            tot[k] += r[k]
    exact = Counter(hashlib.md5(s.encode()).hexdigest() for s in sources)
    near  = Counter(hashlib.md5(normalize_code(s).encode()).hexdigest() for s in sources)
    lens  = sorted(len(s) for s in sources)

    validity_pct = 100 * valid / n
    nl_pct = 100 * (tot["comment_chars"] + tot["docstring_chars"]) / tot["total_chars"]
    comments_pct = 100 * tot["comment_chars"] / tot["total_chars"]
    docstrings_pct = 100 * tot["docstring_chars"] / tot["total_chars"]
    unique_near_pct = 100 * len(near) / n
    len_mean = statistics.mean(lens)
    len_median = statistics.median(lens)

    result = {
        "label": label, "n": n,
        "validity_pct": validity_pct,
        "nl_pct": nl_pct,
        "comments_pct": comments_pct,
        "docstrings_pct": docstrings_pct,
        "unique_near_pct": unique_near_pct,
        "len_mean": len_mean,
        "len_median": len_median,
    }
    print(f"{label:14s}  validity={validity_pct:5.2f}%  "
          f"NL={nl_pct:5.2f}%  unique_near={unique_near_pct:5.2f}%  "
          f"mean_len={len_mean:.0f}")
    return result


print("definitions ready")

definitions ready


## Load 5 slices spread across the corpus

Uses non-streaming load so `.select(range(...))` can jump to arbitrary positions
(streaming can only go forward from the start). For a 1.8M-row dataset this
downloads the full parquet files once (same cost as any full load), then slices
are free.


In [4]:
from datasets import load_dataset

ds = load_dataset(DATASET_ID, split=SPLIT)
n_total = ds.num_rows
print(f"Total documents in corpus: {n_total:,}")

sample = ds[0]
FIELD_NAME = "content" if "content" in sample else list(sample.keys())[0]
print("Using field:", FIELD_NAME)

positions = {
    "start (0%)":   0,
    "25%":          n_total // 4,
    "50%":          n_total // 2,
    "75%":          (3 * n_total) // 4,
    "end (~100%)":  n_total - SLICE_SIZE,
}
print("Slice start indices:", positions)

README.md:   0%|          | 0.00/4.55k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

data/train-00000-of-00019.parquet: reconstructing file:   0%|          |  0.00B /  191MB            

data/train-00000-of-00019.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00019.parquet: reconstructing file:   0%|          |  0.00B /  192MB            

...

data/train-00018-of-00019.parquet: reconstructing file:   0%|          |  0.00B /  188MB            

data/train-00018-of-00019.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1800000 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/19 [00:00<?, ?it/s]

Total documents in corpus: 1,800,000
Using field: content
Slice start indices: {'start (0%)': 0, '25%': 450000, '50%': 900000, '75%': 1350000, 'end (~100%)': 1799000}


## Run the audit on all 5 slices

In [5]:
import json

results = []
for label, start in positions.items():
    end = min(start + SLICE_SIZE, n_total)
    sources = ds[start:end][FIELD_NAME]
    results.append(audit_slice(sources, label))

with open(OUT_JSON, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved: {OUT_JSON}")

start (0%)      validity=85.20%  NL=15.86%  unique_near=99.90%  mean_len=5067
25%             validity=85.20%  NL=14.62%  unique_near=100.00%  mean_len=5100
50%             validity=86.80%  NL=14.93%  unique_near=100.00%  mean_len=5035
75%             validity=84.20%  NL=14.77%  unique_near=100.00%  mean_len=5061
end (~100%)     validity=84.80%  NL=14.92%  unique_near=100.00%  mean_len=5146

Saved: stratified_quality_report.json


## Verdict — is the corpus homogeneous?

In [6]:
nl_vals = [r["nl_pct"] for r in results]
valid_vals = [r["validity_pct"] for r in results]

nl_range = max(nl_vals) - min(nl_vals)
valid_range = max(valid_vals) - min(valid_vals)

print(f"NL% across slices: {min(nl_vals):.2f}% - {max(nl_vals):.2f}%  (range: {nl_range:.2f} points)")
print(f"Validity% across slices: {min(valid_vals):.2f}% - {max(valid_vals):.2f}%  (range: {valid_range:.2f} points)")
print()

if nl_range < 3 and valid_range < 5:
    print("VERDICT: Corpus appears HOMOGENEOUS across position.")
    print("The original 5,000-doc sample (drawn from the start) is representative")
    print("of the full 1.8M-document corpus within normal sampling variation.")
else:
    print("VERDICT: Meaningful variation detected across corpus position.")
    print("The original sample may not generalize evenly - consider reporting")
    print("the range, or use a stratified sample for the paper's headline numbers.")

NL% across slices: 14.62% - 15.86%  (range: 1.24 points)
Validity% across slices: 84.20% - 86.80%  (range: 2.60 points)

VERDICT: Corpus appears HOMOGENEOUS across position.
The original 5,000-doc sample (drawn from the start) is representative
of the full 1.8M-document corpus within normal sampling variation.


## Notes

- **Why 5 slices of 1,000 instead of 1 slice of 5,000:** this tests the specific
  concern "does document position in the corpus correlate with quality/NL
  content?" — a single large sample from one location can't answer that; five
  samples from different locations can.
- **Interpretation:** ranges under ~2-3 percentage points across slices are
  normal sampling noise, not a real trend — do not over-interpret small
  fluctuations as meaningful heterogeneity.
- **If the verdict is "homogeneous"** (expected, since one cleaning function
  was applied uniformly to the whole stream), you can cite this stratified
  check as evidence that the paper's 5,000-doc headline numbers generalize,
  e.g.: *"A stratified check across five 1,000-document samples spanning the
  corpus found natural-language content within Xpp of the reported 16.0% at
  every position, confirming the sample is representative."*
- This does **not** replace the main `pretraining_data_quality.ipynb` audit —
  it's a targeted robustness check on top of it.
